# Adult cencus income dataset prediction with LightGBM

## Load dataset

In [15]:
import pandas as pd

# column names
columns = [
    'age', 'workclass', 'fnlwgt', 'education',
    'education-num', 'marital-status', 'occupation',
    'relationship', 'race', 'sex', 'capital-gain',
    'capital-loss', 'hours-per-week', 'native-country',
    'income'
]

# load dataset
df = pd.read_csv(
    'adult.data',
    header=None,
    names=columns,
    skipinitialspace=True
)

print("Dataset shape:", df.shape)

Dataset shape: (32561, 15)


In [16]:
X = df.drop(columns='income')
y = df['income']

# encode target
y = (y == '>50K').astype(int)

In [21]:
from sklearn.model_selection import train_test_split

# split train/valid/test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full
)


# categorical columns
cat_cols = X.select_dtypes(include=['object', 'str']).columns

# convert to category
for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')
    X_test[col] = X_test[col].astype('category')

## Train model

In [22]:
import lightgbm as lgb

model = lgb.LGBMClassifier(
    n_estimators=5000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    random_state=42,
    verbose=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[lgb.early_stopping(50),lgb.log_evaluation(100)]
)

print('Best iteration:', model.best_iteration_)

/Users/hofman/Documents/najot_talim/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.286451
Early stopping, best iteration is:
[137]	valid_0's binary_logloss: 0.285125
Best iteration: 137


## Evaluate model

In [23]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,roc_auc_score,log_loss

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_prob))
print('Log Loss:', log_loss(y_test, y_prob))

print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))

print('\nClassification Report:')
print(classification_report(y_test, y_pred))

Accuracy: 0.874251497005988
ROC-AUC: 0.9287054280761848
Log Loss: 0.27810838836964447

Confusion Matrix:
[[4640  305]
 [ 514 1054]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      4945
           1       0.78      0.67      0.72      1568

    accuracy                           0.87      6513
   macro avg       0.84      0.81      0.82      6513
weighted avg       0.87      0.87      0.87      6513



In [24]:
from sklearn.metrics import log_loss, accuracy_score

leaves_values = [4, 16, 64, 256]

results = []

for leaves in leaves_values:

    model = lgb.LGBMClassifier(
        n_estimators=5000,
        learning_rate=0.05,
        num_leaves=leaves,
        max_depth=-1,
        min_child_samples=20,
        random_state=42,
        verbose=-1
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )

    train_prob = model.predict_proba(X_train)[:, 1]
    valid_prob = model.predict_proba(X_valid)[:, 1]

    results.append({
        'num_leaves': leaves,
        'best_iteration': model.best_iteration_,
        'train_logloss': log_loss(y_train, train_prob),
        'valid_logloss': log_loss(y_valid, valid_prob),
        'train_accuracy': accuracy_score(y_train, model.predict(X_train)),
        'valid_accuracy': accuracy_score(y_valid, model.predict(X_valid))
    })

results_df = pd.DataFrame(results)

results_df

/Users/hofman/Documents/najot_talim/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[907]	valid_0's binary_logloss: 0.288244
Training until validation scores don't improve for 50 rounds


/Users/hofman/Documents/najot_talim/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Early stopping, best iteration is:
[211]	valid_0's binary_logloss: 0.287343
Training until validation scores don't improve for 50 rounds


/Users/hofman/Documents/najot_talim/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Early stopping, best iteration is:
[126]	valid_0's binary_logloss: 0.287506
Training until validation scores don't improve for 50 rounds


/Users/hofman/Documents/najot_talim/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Early stopping, best iteration is:
[64]	valid_0's binary_logloss: 0.301957


,num_leaves,best_iteration,train_logloss,valid_logloss,train_accuracy,valid_accuracy
0,4,907,0.261692,0.288244,0.880887,0.868857
1,16,211,0.252602,0.287343,0.886978,0.871468
2,64,126,0.218016,0.287506,0.905303,0.871314
3,256,64,0.193802,0.301957,0.929719,0.862408


# Presentationdagi amaliy topshiriqlar

In [25]:
import time
import lightgbm as lgb
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    log_loss
)


# 2. GENERATE DATASET: 250,000 ROWS, 50 FEATURES
X, y = make_classification(
    n_samples=250_000,
    n_features=50,
    n_informative=30,
    n_redundant=10,
    n_classes=2,
    random_state=42
)

print("Dataset shape:", X.shape)


# 3. TRAIN / VALIDATION / TEST SPLIT
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test:", X_test.shape)

Dataset shape: (250000, 50)
Train: (150000, 50)
Valid: (50000, 50)
Test: (50000, 50)


# 4. XGBOOST HIST VS LIGHTGBM

In [26]:
# 4. XGBOOST HIST VS LIGHTGBM
#    TRAINING TIME AND ROC-AUC

results = []

# XGBoost HIST
xgb_model = xgb.XGBClassifier(
    n_estimators=5000,
    learning_rate=0.05,
    max_depth=6,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

start = time.time()

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

xgb_time = time.time() - start

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

results.append({
    "Model": "XGBoost hist",
    "Training time": xgb_time,
    "ROC-AUC": roc_auc_score(y_test, xgb_prob),
    "Best iteration": xgb_model.best_iteration
})


# LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=5000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

start = time.time()

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(0)
    ]
)

lgb_time = time.time() - start

lgb_prob = lgb_model.predict_proba(X_test)[:, 1]

results.append({
    "Model": "LightGBM",
    "Training time": lgb_time,
    "ROC-AUC": roc_auc_score(y_test, lgb_prob),
    "Best iteration": lgb_model.best_iteration_
})

comparison_df = pd.DataFrame(results)

print("XGBOOST VS LIGHTGBM")
print(comparison_df)

/Users/hofman/Documents/najot_talim/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2759]	valid_0's binary_logloss: 0.0597199
XGBOOST VS LIGHTGBM
          Model  Training time   ROC-AUC  Best iteration
0  XGBoost hist      11.717432  0.995065            2344
1      LightGBM      21.970187  0.994969            2759


# 5. XGBOOST EXACT VS HIST

In [27]:
# 5. XGBOOST EXACT VS HIST
#    50,000 ROWS

X_small = X_train[:50_000]
y_small = y_train[:50_000]

exact_results = []

for method in ["exact", "hist"]:

    model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        tree_method=method,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    start = time.time()

    model.fit(X_small, y_small, verbose=False)

    elapsed = time.time() - start

    exact_results.append({
        "tree_method": method,
        "training_time": elapsed
    })

exact_df = pd.DataFrame(exact_results)

print("EXACT VS HIST")
print(exact_df)

exact_time = exact_df.loc[
    exact_df["tree_method"] == "exact",
    "training_time"
].iloc[0]

hist_time = exact_df.loc[
    exact_df["tree_method"] == "hist",
    "training_time"
].iloc[0]

print(
    "\nExact is",
    round(exact_time / hist_time, 2),
    "times slower than hist"
)

EXACT VS HIST
  tree_method  training_time
0       exact       3.297801
1        hist       0.653789

Exact is 5.04 times slower than hist


# 6. LIGHTGBM NUM_LEAVES EXPERIMENT

In [28]:
# 6. LIGHTGBM NUM_LEAVES EXPERIMENT
#    4, 16, 64, 256

leaves_values = [4, 16, 64, 256]

leaves_results = []

for leaves in leaves_values:

    model = lgb.LGBMClassifier(
        n_estimators=5000,
        learning_rate=0.05,
        num_leaves=leaves,
        max_depth=-1,
        min_child_samples=20,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )

    model.fit(
        X_train, y_train,
        eval_set=[
            (X_train, y_train),
            (X_valid, y_valid)
        ],
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(0)
        ]
    )

    train_prob = model.predict_proba(X_train)[:, 1]
    valid_prob = model.predict_proba(X_valid)[:, 1]

    leaves_results.append({
        "num_leaves": leaves,
        "best_iteration": model.best_iteration_,
        "train_logloss": log_loss(y_train, train_prob),
        "valid_logloss": log_loss(y_valid, valid_prob),
        "train_accuracy": accuracy_score(
            y_train, model.predict(X_train)
        ),
        "valid_accuracy": accuracy_score(
            y_valid, model.predict(X_valid)
        )
    })

leaves_df = pd.DataFrame(leaves_results)

print("NUM_LEAVES RESULTS")
print(leaves_df)

/Users/hofman/Documents/najot_talim/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[5000]	training's binary_logloss: 0.0929451	valid_1's binary_logloss: 0.108249


/Users/hofman/Documents/najot_talim/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[4779]	training's binary_logloss: 0.00972662	valid_1's binary_logloss: 0.061872


/Users/hofman/Documents/najot_talim/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1402]	training's binary_logloss: 0.00598504	valid_1's binary_logloss: 0.058568


/Users/hofman/Documents/najot_talim/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[420]	training's binary_logloss: 0.00418712	valid_1's binary_logloss: 0.0557754
NUM_LEAVES RESULTS
   num_leaves  best_iteration  train_logloss  valid_logloss  train_accuracy  \
0           4            5000       0.092945       0.108249        0.981087   
1          16            4779       0.009727       0.061872        0.999900   
2          64            1402       0.005985       0.058568        1.000000   
3         256             420       0.004187       0.055775        1.000000   

   valid_accuracy  
0         0.97464  
1         0.98862  
2         0.98936  
3         0.98908  


# 7. EARLY STOPPING: BEST ITERATION

In [29]:
# 7. EARLY STOPPING: BEST ITERATION

print("EARLY STOPPING")

print(
    "XGBoost best iteration:",
    xgb_model.best_iteration
)

print(
    "LightGBM best iteration:",
    lgb_model.best_iteration_
)

print(
    "LightGBM best score:",
    lgb_model.best_score_
)

print(
    "XGBoost best score:",
    xgb_model.best_score
)

EARLY STOPPING
XGBoost best iteration: 2344
LightGBM best iteration: 2759
LightGBM best score: defaultdict(<class 'collections.OrderedDict'>, {'valid_0': OrderedDict({'binary_logloss': np.float64(0.05971986711012346)})})
XGBoost best score: 0.06027567293426144


# 8. XGBOOST GROW POLICY

In [30]:
# 8. XGBOOST GROW POLICY
#    DEPTHWISE VS LOSSGUIDE


grow_results = []

for policy in ["depthwise", "lossguide"]:

    model = xgb.XGBClassifier(
        n_estimators=5000,
        learning_rate=0.05,
        max_depth=0 if policy == "lossguide" else 6,
        max_leaves=31,
        grow_policy=policy,
        tree_method="hist",
        eval_metric="logloss",
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1
    )

    start = time.time()

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    elapsed = time.time() - start

    y_prob = model.predict_proba(X_test)[:, 1]

    grow_results.append({
        "grow_policy": policy,
        "training_time": elapsed,
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "best_iteration": model.best_iteration
    })

grow_df = pd.DataFrame(grow_results)

print("GROW POLICY RESULTS")
print(grow_df)

GROW POLICY RESULTS
  grow_policy  training_time   ROC-AUC  best_iteration
0   depthwise      14.875906  0.995008            3298
1   lossguide      34.039416  0.995004            3131


# FINAL TEST

In [31]:
# 9. FINAL TEST EVALUATION


print("FINAL XGBOOST EVALUATION")

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("Log Loss:", log_loss(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


print("FINAL LIGHTGBM EVALUATION")

y_pred_lgb = lgb_model.predict(X_test)
y_prob_lgb = lgb_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_lgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lgb))
print("Log Loss:", log_loss(y_test, y_prob_lgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lgb))



# 10. FINAL RESULTS SUMMARY
print("FINAL RESULTS")

print("\n1. XGBoost vs LightGBM:")
print(comparison_df.to_string(index=False))

print("\n2. Exact vs Hist:")
print(exact_df.to_string(index=False))

print("\n3. LightGBM num_leaves:")
print(leaves_df.to_string(index=False))

print("\n4. XGBoost grow_policy:")
print(grow_df.to_string(index=False))

FINAL XGBOOST EVALUATION
Accuracy: 0.98898
ROC-AUC: 0.9950651678446164
Log Loss: 0.057579148560762405

Confusion Matrix:
[[24666   345]
 [  206 24783]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     25011
           1       0.99      0.99      0.99     24989

    accuracy                           0.99     50000
   macro avg       0.99      0.99      0.99     50000
weighted avg       0.99      0.99      0.99     50000

FINAL LIGHTGBM EVALUATION
Accuracy: 0.98892
ROC-AUC: 0.9949686502259306
Log Loss: 0.057833388314658285

Confusion Matrix:
[[24682   329]
 [  225 24764]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     25011
           1       0.99      0.99      0.99     24989

    accuracy                           0.99     50000
   macro avg       0.99      0.99      0.99     50000
weighted avg       0.99      0.99      0.99     